# LSTM Time-Series Forecasting (Google Colab)

End-to-end pipeline for **electricity total load** forecasting using historical CSV data:

1. Exploratory Data Analysis (EDA)
2. Data preprocessing & sliding-window sequences
3. Stacked LSTM training (TensorFlow/Keras)
4. Evaluation (MAE, RMSE, R²)
5. Model export & inference demo

**Runtime:** Enable **GPU** in Colab (*Runtime → Change runtime type → GPU*) for faster training.

**Data:** Upload `GUI_TOTAL_LOAD_DAYAHEAD_202412312300-202512312300.csv` or mount Google Drive.

## 1. Setup & dependencies

In [ ]:
!pip install -q pandas numpy matplotlib scikit-learn tensorflow joblib

In [ ]:
import json
import os
import zipfile
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", bool(gpus), gpus)

## 2. Load dataset

Set `DATA_SOURCE` to `"upload"` or `"drive"`, then run the cells below.

In [ ]:
DATA_SOURCE = "upload"  # "upload" or "drive"
DRIVE_CSV_PATH = "/content/drive/MyDrive/GUI_TOTAL_LOAD_DAYAHEAD_202412312300-202512312300.csv"

if DATA_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = Path("/content") / next(iter(uploaded.keys()))
elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_PATH = Path(DRIVE_CSV_PATH)
else:
    raise ValueError('DATA_SOURCE must be "upload" or "drive"')

assert DATA_PATH.exists(), f"File not found: {DATA_PATH}"
print("Data:", DATA_PATH)

In [ ]:
import shutil

PROJECT_ROOT = Path("/content/lstm_forecast")
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
EDA_DIR = OUTPUT_DIR / "eda"
for d in [PROJECT_ROOT, OUTPUT_DIR, MODEL_DIR, ARTIFACT_DIR, EDA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

project_csv = PROJECT_ROOT / DATA_PATH.name
if DATA_PATH.resolve() != project_csv.resolve():
    shutil.copy(DATA_PATH, project_csv)
DATA_PATH = project_csv
print("Using:", DATA_PATH)

## 3. Configuration

Adjust `HORIZON`, `FAST_MODE`, and `MAX_ROWS` here.

In [ ]:
# --- User settings ---
HORIZON = "1d"          # "1d" | "7d" | "month" | "year"
FAST_MODE = True        # True: 12k rows + 12 epochs (quick); False: full data
RANDOM_SEED = 42

# Column mapping
TIME_COL = "MTU (CET/CEST)"
TARGET_COL = "Actual Total Load (MW)"
FEATURE_COLS = ["Day-ahead Total Load Forecast (MW)"]

HORIZON_STEPS = {"1d": 96, "7d": 672, "month": 720, "year": 7}
LOOKBACK_STEPS = {"1d": 672, "7d": 2016, "month": 336, "year": 30}
RESAMPLE_RULE = {"1d": None, "7d": None, "month": "1h", "year": "1D"}

TRAIN_RATIO, VAL_RATIO = 0.70, 0.15
LSTM_UNITS = (64, 32)
DROPOUT = 0.2
LEARNING_RATE = 1e-3
BATCH_SIZE = 128
EPOCHS = 12 if FAST_MODE else 30
PATIENCE = 10
MAX_ROWS = 12000 if FAST_MODE else None

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print(f"Horizon={HORIZON}, epochs={EPOCHS}, max_rows={MAX_ROWS}")

## 4. Data loading & preprocessing functions

In [ ]:
def parse_timestamp(mtu_series):
    starts = mtu_series.str.split(" - ").str[0]
    starts = starts.str.replace(r"\s*\((CET|CEST)\)\s*$", "", regex=True)
    return pd.to_datetime(starts, format="%d/%m/%Y %H:%M", dayfirst=True)


def load_raw_data(csv_path):
    df = pd.read_csv(csv_path)
    df["timestamp"] = parse_timestamp(df[TIME_COL])
    df = df.sort_values("timestamp").drop_duplicates(subset=["timestamp"])
    df = df.set_index("timestamp")
    numeric_cols = [TARGET_COL] + FEATURE_COLS
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df[numeric_cols].copy()
    hour = df.index.hour + df.index.minute / 60.0
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df.index.dayofweek / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df.index.dayofweek / 7)
    df["doy_sin"] = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * df.index.dayofyear / 365.25)
    if MAX_ROWS is not None:
        df = df.iloc[-MAX_ROWS:]
    return df


def handle_missing_values(df):
    out = df.copy()
    missing_before = int(out.isnull().sum().sum())
    out = out.interpolate(method="time", limit_direction="both").ffill().bfill()
    return out, {"missing_before": missing_before, "missing_after": int(out.isnull().sum().sum())}


def resample_for_horizon(df, horizon):
    rule = RESAMPLE_RULE.get(horizon)
    if rule is None:
        return df
    return df.select_dtypes(include=[np.number]).resample(rule).mean().dropna()


def prepare_series(df, horizon):
    feature_names = [TARGET_COL] + FEATURE_COLS + [
        "hour_sin", "hour_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos"
    ]
    resampled = resample_for_horizon(df, horizon)
    available = [c for c in feature_names if c in resampled.columns]
    return resampled[available], available


def create_sequences(scaled_df, feature_cols, lookback, horizon_steps):
    features = scaled_df[feature_cols].values
    target = scaled_df["scaled_target"].values
    n = len(scaled_df)
    max_start = n - lookback - horizon_steps + 1
    if max_start <= 0:
        raise ValueError(f"Need {lookback + horizon_steps} rows, have {n}")
    X, y = [], []
    for i in range(max_start):
        X.append(features[i : i + lookback])
        y.append(target[i + lookback : i + lookback + horizon_steps])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def build_datasets(csv_path, horizon):
    raw = load_raw_data(csv_path)
    cleaned, missing_info = handle_missing_values(raw)
    series_df, feature_cols = prepare_series(cleaned, horizon)
    lookback = LOOKBACK_STEPS[horizon]
    horizon_steps = HORIZON_STEPS[horizon]
    n = len(series_df)
    train_end = int(n * TRAIN_RATIO)
    val_end = int(n * (TRAIN_RATIO + VAL_RATIO))
    train_df, val_df, test_df = (
        series_df.iloc[:train_end],
        series_df.iloc[train_end:val_end],
        series_df.iloc[val_end:],
    )
    feature_scaler, target_scaler = MinMaxScaler(), MinMaxScaler()
    feature_scaler.fit(train_df[feature_cols])
    target_scaler.fit(train_df[[TARGET_COL]])

    def scale_part(part):
        out = part.copy()
        out[feature_cols] = feature_scaler.transform(part[feature_cols])
        out["scaled_target"] = target_scaler.transform(part[[TARGET_COL]]).ravel()
        return out

    train_s, val_s, test_s = scale_part(train_df), scale_part(val_df), scale_part(test_df)
    X_train, y_train = create_sequences(train_s, feature_cols, lookback, horizon_steps)
    X_val, y_val = create_sequences(val_s, feature_cols, lookback, horizon_steps)
    X_test, y_test = create_sequences(test_s, feature_cols, lookback, horizon_steps)
    metadata = {
        "horizon": horizon,
        "lookback": lookback,
        "horizon_steps": horizon_steps,
        "feature_cols": feature_cols,
        "missing_info": missing_info,
        "n_train": len(X_train),
        "n_val": len(X_val),
        "n_test": len(X_test),
    }
    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_test": X_test, "y_test": y_test,
        "feature_scaler": feature_scaler,
        "target_scaler": target_scaler,
        "metadata": metadata,
        "series_df": series_df,
    }

## 5. Exploratory Data Analysis (EDA)

In [ ]:
df = load_raw_data(DATA_PATH)
print("Shape:", df.shape)
print("Date range:", df.index.min(), "→", df.index.max())
display(df.describe())
print("\nMissing per column:\n", df.isnull().sum())

q1, q3 = df[TARGET_COL].quantile(0.25), df[TARGET_COL].quantile(0.75)
iqr = q3 - q1
outliers = ((df[TARGET_COL] < q1 - 1.5 * iqr) | (df[TARGET_COL] > q3 + 1.5 * iqr)).sum()
print(f"IQR outliers on target: {outliers}")
if FEATURE_COLS[0] in df.columns:
    print(f"Correlation target vs forecast: {df[TARGET_COL].corr(df[FEATURE_COLS[0]]):.4f}")

In [ ]:
sample = df.iloc[:: max(1, len(df) // 5000)]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(sample.index, sample[TARGET_COL], lw=0.6)
axes[0, 0].set_title("Actual Total Load (sampled)")
df[TARGET_COL].hist(bins=50, ax=axes[0, 1], edgecolor="black")
axes[0, 1].set_title("Distribution")
df.groupby(df.index.hour)[TARGET_COL].mean().plot(kind="bar", ax=axes[1, 0], color="steelblue")
axes[1, 0].set_title("Avg load by hour")
df.groupby(df.index.month)[TARGET_COL].mean().plot(kind="bar", ax=axes[1, 1], color="coral")
axes[1, 1].set_title("Avg load by month")
plt.tight_layout()
plt.savefig(EDA_DIR / "eda_overview.png", dpi=120)
plt.show()

## 6. Build datasets & inspect sequences

In [ ]:
data = build_datasets(DATA_PATH, HORIZON)
meta = data["metadata"]
print(json.dumps(meta, indent=2))
print(f"X_train: {data['X_train'].shape}, y_train: {data['y_train'].shape}")

## 7. LSTM model architecture

In [ ]:
def build_lstm_model(lookback, n_features, horizon_steps):
    """Stacked LSTMs + dropout + multi-step linear head (regression)."""
    inputs = keras.Input(shape=(lookback, n_features))
    x = inputs
    for i, units in enumerate(LSTM_UNITS):
        x = layers.LSTM(units, return_sequences=i < len(LSTM_UNITS) - 1)(x)
        x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(horizon_steps, activation="linear")(x)
    model = keras.Model(inputs, outputs, name="lstm_load_forecaster")
    model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss="mse", metrics=["mae"])
    return model


lookback = meta["lookback"]
horizon_steps = meta["horizon_steps"]
n_features = data["X_train"].shape[2]
model = build_lstm_model(lookback, n_features, horizon_steps)
model.summary()

## 8. Training (early stopping + checkpoint)

In [ ]:
model_path = str(MODEL_DIR / f"lstm_{HORIZON}_best.keras")
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(model_path, monitor="val_loss", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
]

history = model.fit(
    data["X_train"], data["y_train"],
    validation_data=(data["X_val"], data["y_val"]),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

# Save scalers for production inference
joblib.dump(data["feature_scaler"], ARTIFACT_DIR / f"lstm_{HORIZON}_feature_scaler.joblib")
joblib.dump(data["target_scaler"], ARTIFACT_DIR / f"lstm_{HORIZON}_target_scaler.joblib")
with open(ARTIFACT_DIR / f"lstm_{HORIZON}_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Saved model:", model_path)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss (MSE)")
axes[0].legend()
axes[1].plot(history.history["mae"], label="train")
axes[1].plot(history.history["val_mae"], label="val")
axes[1].set_title("MAE (scaled)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Evaluation (MAE, RMSE, R²)

In [ ]:
def inverse_transform(y_scaled, target_scaler):
    flat = y_scaled.reshape(-1, 1)
    return target_scaler.inverse_transform(flat).reshape(y_scaled.shape)


best_model = keras.models.load_model(model_path)
y_pred_s = best_model.predict(data["X_test"], verbose=0)
y_true = inverse_transform(data["y_test"], data["target_scaler"])
y_pred = inverse_transform(y_pred_s, data["target_scaler"])

metrics = {
    "MAE": float(mean_absolute_error(y_true.ravel(), y_pred.ravel())),
    "RMSE": float(np.sqrt(mean_squared_error(y_true.ravel(), y_pred.ravel()))),
    "R2": float(r2_score(y_true.ravel(), y_pred.ravel())),
}
print(json.dumps(metrics, indent=2))
print(f"\nMAE: {metrics['MAE']:,.2f} MW | RMSE: {metrics['RMSE']:,.2f} MW | R²: {metrics['R2']:.4f}")

In [ ]:
n_show = min(500, len(y_true))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y_true[:n_show, 0], label="Actual")
axes[0].plot(y_pred[:n_show, 0], label="Predicted", alpha=0.8)
axes[0].set_title("Test set — first forecast step")
axes[0].legend()
idx = len(y_true) // 2
axes[1].plot(y_true[idx], "o-", label="Actual")
axes[1].plot(y_pred[idx], "s-", label="Predicted")
axes[1].set_title(f"Multi-step sample ({horizon_steps} steps)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 10. Inference demo (real-time forecast from latest history)

In [ ]:
series = data["series_df"]
feature_cols = meta["feature_cols"]
recent = series.iloc[-lookback:].copy()
recent[feature_cols] = data["feature_scaler"].transform(recent[feature_cols])
recent["scaled_target"] = data["target_scaler"].transform(recent[[TARGET_COL]]).ravel()
X_live = recent[feature_cols].values.reshape(1, lookback, len(feature_cols))
forecast_scaled = best_model.predict(X_live, verbose=0)
forecast_mw = inverse_transform(forecast_scaled, data["target_scaler"])[0]

print(f"Last timestamp: {series.index[-1]}")
print(f"Forecast length: {len(forecast_mw)} steps")
print(f"First 5 values (MW): {forecast_mw[:5].round(2)}")

## 11. Download artifacts

Zip model, scalers, and plots for local deployment.

In [ ]:
zip_path = "/content/lstm_forecast_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in [MODEL_DIR, ARTIFACT_DIR, EDA_DIR, OUTPUT_DIR]:
        if not folder.exists():
            continue
        for f in folder.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(PROJECT_ROOT))

from google.colab import files
files.download(zip_path)

---

### Horizon reference

| `HORIZON` | Resolution | Steps ahead |
|-----------|------------|-------------|
| `1d` | 15 min | 96 (1 day) |
| `7d` | 15 min | 672 (7 days) |
| `month` | Hourly | 720 (~30 days) |
| `year` | Daily | 7 days* |

\*Full 365-day-ahead forecasting requires multiple years of history.

**Production tips:** Set `FAST_MODE = False`, enable GPU, and train each horizon separately by changing `HORIZON`.